# 01 — Sinh dữ liệu tự chơi

Notebook này đọc artifact của `scripts/generate_selfplay.py`; nó không tự viết game loop. Dùng nó để kiểm manifest, checksum, cân bằng X/O, kết quả từng cặp và replay một game raw.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'core').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.engine.interfaces import Player
from core.ml.data_pipeline import iter_raw_games, replay_raw_game

RAW_DIR = ROOT / 'data/raw_logs/w56_selfplay_dataset_v1'
manifest = json.loads((RAW_DIR / 'run_manifest.json').read_text(encoding='utf-8'))
print('Raw directory:', RAW_DIR)
print('Run ID:', manifest['run_id'])

## 1. Kiểm manifest và checksum

Raw JSONL chỉ được tin cậy khi checksum shard khớp manifest.

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

shard_rows = []
for shard in manifest['shards']:
    path = RAW_DIR / shard['path']
    shard_rows.append({
        'shard': shard['path'], 'declared_games': shard['games'],
        'checksum_matches': sha256(path) == shard['sha256'], 'size_bytes': path.stat().st_size,
    })
display(pd.DataFrame(shard_rows))
assert all(row['checksum_matches'] for row in shard_rows)
pd.Series({key: manifest[key] for key in ('requested_games', 'successful_games', 'failed_games', 'base_seed', 'pair_games')})

## 2. Thống kê game và cân bằng bên

Các thống kê dưới đây lấy từ raw game, nên không bị ảnh hưởng bởi dedup hoặc D4 augmentation của dataset tensor.

In [ ]:
records = [record for shard in manifest['shards'] for record in iter_raw_games(RAW_DIR / shard['path'])]
assert len(records) == manifest['successful_games']

stats = {}
for agent in sorted({r.x_agent for r in records} | {r.o_agent for r in records}):
    games = [r for r in records if agent in (r.x_agent, r.o_agent)]
    wins = sum((r.winner == Player.X and r.x_agent == agent) or (r.winner == Player.O and r.o_agent == agent) for r in games)
    draws = sum(r.winner is None for r in games)
    stats[agent] = {'games': len(games), 'wins': wins, 'losses': len(games) - wins - draws, 'draws': draws,
                    'as_x': sum(r.x_agent == agent for r in games), 'as_o': sum(r.o_agent == agent for r in games)}
agent_stats = pd.DataFrame(stats).T
display(agent_stats)
agent_stats[['wins', 'losses', 'draws']].plot(kind='bar', figsize=(8, 4), ylabel='Games', rot=0)
plt.title('Kết quả raw self-play theo agent')
plt.show()

In [ ]:
pair_rows = []
for left, right in manifest['pairs']:
    games = [r for r in records if {r.x_agent, r.o_agent} == {left, right}]
    pair_rows.append({'pair': f'{left} vs {right}', 'games': len(games),
                      f'{left} as X': sum(r.x_agent == left for r in games),
                      f'{right} as X': sum(r.x_agent == right for r in games)})
pair_summary = pd.DataFrame(pair_rows)
display(pair_summary)
assert (pair_summary.iloc[:, 2] == pair_summary.iloc[:, 3]).all(), 'X/O side imbalance detected'

## 3. Replay audit

Mỗi game phải tái dựng được từ bàn trống đến terminal. Cell này chạy lại validator canonical cho toàn bộ raw record.

In [ ]:
for record in records:
    replay_raw_game(record)
print(f'Replay thành công {len(records)}/{len(records)} game; không có action invalid.')

## 4. Handoff

Sau khi checksum và replay xanh, bước kế tiếp là build tensor bằng `scripts/build_dataset.py`, sau đó M1 dùng `load_dataset_split()` để train PolicyNet. Không dùng raw JSONL trực tiếp trong train loop.